# Blackberry 3DGS on Google Colab (`engines_cuda`)

Uses **PyTorch + gsplat** (CUDA). PyTorch3D is optional (mesh tools) — see last section.

**Runtime → GPU** required.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Clone repo + install deps

Uses branch `feature/cuda-torch-train` (or `feature/colab-train` once merged).

In [ ]:
import os
REPO = "/content/splats"
BRANCH = "feature/colab-train"
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} --single-branch https://github.com/aidazsolt-bot/splats.git {REPO}
else:
    %cd {REPO}
    !git fetch origin {BRANCH} && git checkout {BRANCH} && git pull
%cd {REPO}
# gsplat builds CUDA extensions against the Colab torch/CUDA
!pip -q install -U pip
!pip -q install gsplat numpy pillow scipy tqdm imageio open3d
import gsplat, torch
print("gsplat OK", gsplat.__version__ if hasattr(gsplat, "__version__") else "", "cuda", torch.cuda.is_available())

## 3. Mount Drive and set dataset path

Expected layout:
```
MyDrive/blackberry-splats/dataset/{cameras,images,points3D}.txt + images/
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/blackberry-splats/dataset"
OUT = "/content/drive/MyDrive/blackberry-splats/engines_cuda_out"

import os
assert os.path.isfile(f"{DATA}/cameras.txt"), f"Missing COLMAP at {DATA}"
assert os.path.isdir(f"{DATA}/images"), f"Missing images/ at {DATA}"
os.makedirs(OUT, exist_ok=True)
print("DATA", DATA)
print("OUT ", OUT)
!ls "{DATA}" | head
!ls "{DATA}/images" | wc -l

## 4. Smoke train (optional)

In [ ]:
%cd /content/splats
!python engines_cuda/run_train.py --smoke --data "{DATA}" --out "{OUT}/smoke" --data-factor 4

## 5. Full train (photorealistic)

T4: start with `--data-factor 4` or `2`. A100 can use `2` or `1`.

After a disconnect: re-run with the **same** flags plus `--resume` (loads `ckpt_latest.pt` from `OUT`).

In [ ]:
%cd /content/splats
RESUME = False  # set True to continue after Colab disconnect
cmd = (
    f'python engines_cuda/run_train.py --data "{DATA}" --out "{OUT}" '
    f'--data-factor 2 --max-steps 30000'
)
if RESUME:
    cmd += " --resume"
!{cmd}

## 6. List outputs / optional mesh from checkpoint

In [ ]:
!ls -lh "{OUT}"
# Optional mesh bake from last ckpt:
# !python engines_cuda/export/mesh_from_ckpt.py --ckpt "{OUT}/ckpt_30000.pt" --out "{OUT}/mesh_from_splat.ply"

## Optional: PyTorch3D (meshes — not required for 3DGS)

Colab *can* install PyTorch3D; pin to the runtime’s torch/CUDA. Example (may need adjusting when Colab upgrades torch):

In [ ]:
# Optional — skip for Gaussian training
# import torch, sys
# print(torch.__version__, torch.version.cuda)
# !pip -q install --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt251/download.html
# import pytorch3d
# print("pytorch3d", pytorch3d.__version__)